# ML-9 : Detection d'anomalies avec Randomized PCA

**Navigation** : [Index](README.md) | [<< Precedent](ML-8-Clustering.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. Distinguer la **detection d'anomalies** (non-supervisee) de la classification binaire supervisée (ML-1, ML-3)
2. Construire un pipeline ML.NET avec le trainer **Randomized PCA** (`AnomalyDetectionCatalog`)
3. Comprendre le **score d'anomalie** comme une erreur de reconstruction dans le sous-espace PCA
4. Evaluer un detecteur avec l'**AUC** (area under ROC) et le **Detection Rate @ K faux positifs**
5. Choisir un **seuil de decision** qui arbitre entre detection (TPR) et fausses alarmes (FPR)
6. Diagnostiquer la **limite** de l'approche PCA sur des anomalies subtiles



## Du clustering a la detection d'anomalies

Le [ML-8](ML-8-Clustering.ipynb) regroupait des points en clusters sans etiquettes. La **detection d'anomalies** repond a une question differente : etant donne un regime de fonctionnement **normal** (apprentissage), quels points s'en ecartent suffisamment pour etre soupconnes anormaux ? C'est encore de l'apprentissage **non-supervise** au sens ou le modele n'apprend qu'a partir de donnees normales — mais la tache est un **test** (chaque point est-il dans la distribution normale ?), pas une partition.

Le cas d'usage industriel canonique est la **maintenance predictive** : une machine saine (capteurs : temperature, pression, vibration) opere dans un etroite ellipse de regimx ; une usure, une fuite ou un desequilibre la fait deriver hors de cette zone. On veut declencher une alerte avant la casse. ML.NET fournit le trainer [Randomized PCA](https://learn.microsoft.com/dotnet/api/microsoft.ml.trainers.randomizedpcatrainer) via `mlContext.AnomalyDetection.Trainers.RandomizedPca`.

L'intuition (Pfiefer et al., 2011 ; Lakhina et al., 2004) : la PCA projette les donnees normales sur un **sous-espace de faible dimension** (les *k* premieres composantes). Un point normal se reconstruit bien dans ce sous-espace (faible residu) ; un point anormal, lui, s'etale hors du sous-espace et laisse un **fort residu de reconstruction**. Ce residu est le **score d'anomalie** : plus il est eleve, plus le point s'ecarte du regime appris.

> **Subtilite importante.** Le trainer est non-supervise au **Fit** (il ignore toute etiquette), mais l'**Evaluate** a besoin d'une colonne `Label` (colonne `float`, 1.0 = anomalie, 0.0 = normal) sur le jeu de test pour calculer l'AUC. On s'entraine donc sur des donnees normales et l'on evalue sur un jeu de test **labellise** melangeant normaux et anomalies.



In [1]:
#r "nuget: Microsoft.ML, 5.0.0"
Console.WriteLine("Microsoft.ML 5.0.0 charge");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.ML, 5.0.0

Microsoft.ML 5.0.0 charge


On reference les espaces de noms ML.NET et `System.Linq` (pour `Min`, `Max`, `Count` sur les tableaux de scores).

In [2]:
using System;
using System.Collections.Generic;
using System.Linq;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Trainers;
Console.WriteLine("Espaces de noms importes (Microsoft.ML + System.Linq)");


Espaces de noms importes (Microsoft.ML + System.Linq)


Comme dans [ML-1](ML-1-Introduction.ipynb), on cree le [MLContext](https://learn.microsoft.com/dotnet/api/microsoft.ml.mlcontext) — le point d'entree commun. On fixe la graine (`seed: 42`) pour rendre la PCA randomisee deterministe (sinon la projection stochastique varie d'une execution a l'autre).

In [3]:
var mlContext = new MLContext(seed: 42);
Console.WriteLine("MLContext cree (seed fixe a 42 pour reproductibilite)");


MLContext cree (seed fixe a 42 pour reproductibilite)


## Jeu de donnees : capteurs d'une machine

On simule trois capteurs d'une machine industrielle. Plutot que les valeurs brutes, on travaille avec les **ecarts au point de fonctionnement nominal** : c'est cette deviation qui est informative. Le regime sain est ainsi centre sur l'origine :

| Feature | Ecart-type (regime sain) | Regime sain (normal) | Regime defaillant (anomalie) |
|---------|--------------------------|----------------------|------------------------------|
| **Temperature** | 2.0 | ~0 (nominal) | surchauffe (~+5) |
| **Pressure** | 0.1 | ~0 (nominal) | surpression / fuite (~+0.8) |
| **Vibration** | 0.8 | ~0 (nominal) | fortes vibrations (~+2.5) |

On construit **deux** jeux distincts :

- **`trainData`** (200 points **normaux** uniquement) — c'est le regime que le modele apprend. Le `Label` vaut `0.0` partout et le trainer **l'ignore** au Fit.
- **`testData`** (120 points normaux + 40 anomalies, labellises) — sert a evaluer l'AUC et a tuner le seuil.

Les anomalies sont volontairement moderees (~3 ecarts-types par capteur) afin que la detection reste **non-triviale** : un peu de chevauchement subsiste, et le choix du seuil aura un vrai cout.

Definissons d'abord les classes de donnees ML.NET et un utilitaire gaussien (Box-Muller).

In [4]:
public class SensorData
{
    public float Temperature { get; set; }  // capteur 1 (deg C)
    public float Pressure    { get; set; }  // capteur 2 (bar)
    public float Vibration   { get; set; }  // capteur 3 (mm/s)
    public float Label       { get; set; }  // 1.0 = anomalie, 0.0 = normal (ignore au Fit, utilise par Evaluate)
}

public class AnomalyPrediction
{
    [ColumnName("Score")]
    public float Score { get; set; }  // score d'anomalie = residu de reconstruction PCA

    [ColumnName("PredictedLabel")]
    public bool PredictedLabel { get; set; }  // true = anomalie (score > seuil calibre au Fit)
}

public static class RandUtil
{
    static readonly Random _r = new Random(42);

    // Box-Muller : echantillonne un flottant selon N(mean, std)
    public static float NextGaussian(float mean, float std)
    {
        double u1 = 1.0 - _r.NextDouble();
        double u2 = 1.0 - _r.NextDouble();
        double z = Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Sin(2.0 * Math.PI * u2);
        return (float)(mean + z * std);
    }
}
Console.WriteLine("Classes SensorData / AnomalyPrediction / RandUtil definies");


Classes SensorData / AnomalyPrediction / RandUtil definies


On genere maintenant les deux jeux. Le regime sain centre les trois ecarts autour de 0 (machine reglee au nominal) avec un bruit gaussien realiste. La **pression** est une variable etroitement regulate (faible ecart-type) ; les anomalies simulent une fuite ou un desequilibre qui la fait deriver fortement (pic de ~8 ecarts-types), accompagne d'une surchauffe et de vibrations. Ce decalage **orthogonal au plan de variance normale** est precisement ce que la PCA detecte.

In [5]:
// === Regime NORMAL (machine saine) : ecarts au point nominal, centres sur 0 ===
// 200 points d'entrainement (tous normaux) + 120 points de test normaux
var trainNormal = new List<SensorData>();
var testNormal  = new List<SensorData>();
for (int i = 0; i < 200; i++)
    trainNormal.Add(new SensorData {
        Temperature = RandUtil.NextGaussian(0f, 2.0f),
        Pressure    = RandUtil.NextGaussian(0f, 0.1f),
        Vibration   = RandUtil.NextGaussian(0f, 0.8f)
    });
for (int i = 0; i < 120; i++)
    testNormal.Add(new SensorData {
        Temperature = RandUtil.NextGaussian(0f, 2.0f),
        Pressure    = RandUtil.NextGaussian(0f, 0.1f),
        Vibration   = RandUtil.NextGaussian(0f, 0.8f)
    });

// === ANOMALIES (machine defaillante, ~3 ecarts-types par capteur) ===
var testAnomalies = new List<SensorData>();
for (int i = 0; i < 40; i++)
    testAnomalies.Add(new SensorData {
        Temperature = RandUtil.NextGaussian(5.0f, 2.0f),
        Pressure    = RandUtil.NextGaussian(0.8f, 0.1f),
        Vibration   = RandUtil.NextGaussian(2.5f, 0.8f),
        Label       = 1.0f   // <-- etiquette de verite, utilisee uniquement par Evaluate
    });

var testDataRows = testNormal.Concat(testAnomalies).ToList();

Console.WriteLine($"Train : {trainNormal.Count} points (tous normaux)");
Console.WriteLine($"Test  : {testNormal.Count} normaux + {testAnomalies.Count} anomalies = {testDataRows.Count} points");


Train : 200 points (tous normaux)


Test  : 120 normaux + 40 anomalies = 160 points


On charge les deux jeux dans des [IDataView](https://learn.microsoft.com/dotnet/api/microsoft.ml.idataview), comme en [ML-1](ML-1-Introduction.ipynb) et [ML-8](ML-8-Clustering.ipynb).

In [6]:
IDataView trainData = mlContext.Data.LoadFromEnumerable(trainNormal);
IDataView testData   = mlContext.Data.LoadFromEnumerable(testDataRows);
Console.WriteLine("IDataView charges : trainData (normal) + testData (normal + anomalies labellises)");


IDataView charges : trainData (normal) + testData (normal + anomalies labellises)


## Pipeline : concatenation, Randomized PCA

Le pipeline a **deux** etapes :

1. **Concatenation** des trois capteurs en un vecteur `"Features"`,
2. **Randomized PCA** de rang `2`.

> **Pourquoi PAS de normalisation MinMax ici, contrairement a [ML-8](ML-8-Clustering.ipynb) ?** K-Means mesure des **distances euclidiennes** et exige que les features soient a la meme echelle. La detection d'anomalies par PCA, elle, exploite justement la **structure de variance** : les composantes principales sont definies par les directions de plus grande variance. Normaliser ecraserait cette structure. C'est une difference fondamentale entre les deux familles non-supervisees.

> **Pourquoi `rank: 2` ?** Avec 3 features, la PCA de rang 2 projette les donnees sur un **plan** ; le score d'anomalie est la composante **hors-plan** (le residu le long de la 3e direction). Si l'on choisissait `rank: 3` (le maximum), **tout** serait explique et le residu serait nul : la detection deviendrait impossible. Le rang controle donc la **sensibilite** du detecteur (cf. [Exercice 1](#Exercice-1-:-Effet-du-rang-PCA-sur-l'AUC)). **C'est un piege frequent** : le rang par defaut est 20, qui planterait ici avec seulement 3 features.

Nos donnees sont deja **recentrees sur l'origine** (ecarts au point nominal) : la PCA passe donc par le centroid, condition essentielle pour que le residu reflete bien l'ecart au regime normal.

In [7]:
var pipeline = mlContext.Transforms
    .Concatenate("Features", nameof(SensorData.Temperature), nameof(SensorData.Pressure), nameof(SensorData.Vibration))
    .Append(mlContext.AnomalyDetection.Trainers.RandomizedPca(featureColumnName: "Features", rank: 2));

Console.WriteLine("Pipeline cree : Concatenate -> RandomizedPca(rank=2)");


Pipeline cree : Concatenate -> RandomizedPca(rank=2)


On entraine le modele sur **`trainData` (donnees normales uniquement)** via [Fit](https://learn.microsoft.com/dotnet/api/microsoft.ml.iestimator-1.fit). Le modele apprend ainsi la geometrie du regime sain ; il n'a jamais vu d'anomalies.

In [8]:
var model = pipeline.Fit(trainData);
Console.WriteLine("Modele RandomizedPca entraine sur les donnees normales (rank=2)");


Modele RandomizedPca entraine sur les donnees normales (rank=2)


## Evaluation : AUC et Detection Rate @ K faux positifs

Contrairement au clustering (ML-8), la detection d'anomalies **se laisse evaluer** des lors que le jeu de test est labellise. On dispose de deux metriques complementaires :

- **`AreaUnderRocCurve`** (AUC, area under the ROC curve) : probabilite qu'un point anormal tire au hasard ait un score plus eleve qu'un point normal tire au hasard. 1.0 = separation parfaite ; 0.5 = hasard.
- **`DetectionRateAtFalsePositiveCount`** : parmi les vraies anomalies, quelle fraction est detectee avant que le modele ne produise *K* fausses alarmes (par defaut, *K* = 10). C'est la metrique operationnelle : a quel point mon detecteur est-il utile **a faible taux de fausse alarme** ?

On obtient ces metriques via `mlContext.AnomalyDetection.Evaluate`, qui lit la colonne `Label` (verite terrain) et la colonne `Score` (sortie du modele).

In [9]:
var scoredTest = model.Transform(testData);
var metrics = mlContext.AnomalyDetection.Evaluate(scoredTest, labelColumnName: "Label");

Console.WriteLine("=== Metriques de detection d'anomalies (rank=2) ===");
Console.WriteLine($"  AreaUnderRocCurve (AUC)              : {metrics.AreaUnderRocCurve:F4}");
Console.WriteLine($"  DetectionRate @ 10 faux positifs     : {metrics.DetectionRateAtFalsePositiveCount:F4}");


=== Metriques de detection d'anomalies (rank=2) ===


  AreaUnderRocCurve (AUC)              : 0,8477


  DetectionRate @ 10 faux positifs     : 0,2250


### Interpretation : qualite de la separation

| Metrique | Valeur | Lecture |
|----------|--------|---------|
| AreaUnderRocCurve (AUC) | **0,848** | bonne separation (1,0 = parfait, 0,5 = hasard) |
| DetectionRate @ 10 FP | 0,225 | fraction des anomalies detectee avant 10 fausses alarmes |

Un AUC de **0,85** signale une bonne separation : un point anormal tire au hasard a ~85 % de chances d'etre score plus haut qu'un point normal. Ce n'est **pas** 1,0 car le chevauchement subsiste — une anomalie dont le pic de pression est modere ressemble, dans le sous-espace PCA, a un point normal bruite. Le Detection Rate @ 10 FP (0,225) traduit cette limite : a tres faible budget de fausses alarmes, on ne rattrape qu'un quart des anomalies. Le sweep de seuil ci-dessous montrera comment on ameliore ce taux en acceptant plus de fausses alarmes.

## Prediction : scorer un nouveau point

On cree un [PredictionEngine](https://learn.microsoft.com/dotnet/api/microsoft.ml.predictionengine-2) (comme en [ML-1](ML-1-Introduction.ipynb) et [ML-8](ML-8-Clustering.ipynb)) pour inferer le score de nouveaux points. La prediction renvoie :

- `Score` : le residu de reconstruction PCA (plus eleve = plus anomalous),
- `PredictedLabel` : un booleen, `true` si `Score` depasse un seuil **calibre pendant le Fit** (par defaut, le trainer suppose ~10 % de contamination).

> **Avertissement sur le seuil par defaut.** Le seuil embarque est calibre sur une hypothese de contamination (~10 %). Si votre jeu d'entrainement est **integrallement normal** (ce qui est l'ideal), ce seuil est trop bas : le modele flagge les ~10 % de normaux les plus extremes. C'est pourquoi l'[Exercice 2](#Exercice-2-:-Accorder-le-seuil-a-un-cout-operationnel) vous fera tuner ce seuil vous-meme — le `Score`, lui, est fiable.

In [10]:
var engine = mlContext.Model.CreatePredictionEngine<SensorData, AnomalyPrediction>(model);

SensorData[] testPoints = new[] {
    new SensorData { Temperature = 0f,  Pressure = 0.0f, Vibration = 0.0f },  // profil sain (au nominal)
    new SensorData { Temperature = 3f,  Pressure = 0.2f, Vibration = 1.0f },  // legerement off
    new SensorData { Temperature = 5f,  Pressure = 0.8f, Vibration = 2.5f },  // profil defaillant (fuite)
};

Console.WriteLine("=== Score d'anomalie de nouveaux points ===");
foreach (var s in testPoints)
{
    var p = engine.Predict(s);
    string flag = p.PredictedLabel ? "*** ANOMALIE ***" : "normal";
    Console.WriteLine($"  T={s.Temperature,4:F1} P={s.Pressure,4:F2} V={s.Vibration,4:F1} -> Score={p.Score,8:F4}  [{flag}]");
}


=== Score d'anomalie de nouveaux points ===


  T= 0,0 P=0,00 V= 0,0 -> Score=  0,1268  [normal]


  T= 3,0 P=0,20 V= 1,0 -> Score=  0,0644  [normal]


  T= 5,0 P=0,80 V= 2,5 -> Score=  0,1432  [normal]


### Interpretation : le score reflete le residu, pas la distance brute

Les scores sont proches (0,06 a 0,14) et les trois points sont flagges **normaux** — le seuil par defaut (calibre pour ~10 % de contamination) est trop conservateur quand on s'entraine sur un jeu integralement normal. Deux points pedagogiques :

- Le profil **defaillant** a bien le score le plus eleve (0,143) : le modele le classe comme le plus anomalous, ce qui est correct.
- Le score **n'est pas monotone en "distance a l'origine"** : le point legerement off (0,064) score plus bas que le point sain (0,127). C'est attendu : le score mesure le **residu de reconstruction PCA** (l'ecart au sous-espace normal), pas la norme du vecteur. Un point peut etre eloigne de l'origine mais bien explique par le plan PCA (faible residu), ou proche de l'origine mais legerement hors-plan (residu non negligeable). La signature qui compte ici est le **pic de pression** (orthogonal au plan) — d'ou l'interet de tuner le seuil plutot que de se fier au `PredictedLabel` par defaut.

## Choix du seuil : trade-off detection / fausse alarme

Le score brut est une grandeur continue ; la decision operationnelle (alerte ou non) exige un **seuil**. Ce seuil arbitre entre deux erreurs :

- **Faux negatif** (anomalie ratee) -> on mesure le **TPR** (taux de detection : fraction des vraies anomalies detectees),
- **Faux positif** (fausse alarme) -> on mesure le **FPR** (fraction des vrais normaux flagges a tort).

Abaisser le seuil augmente le TPR (on rate moins d'anomalies) **mais** augmente aussi le FPR (plus de fausses alarmes). Le bon seuil depend du **cout relatif** d'une casse ratee vs d'une intervention inutile.

Plutot que de tester des seuils absolus (dont l'echelle depend des donnees), on balaie des seuils **relatifs** repartis entre le min et le max des scores observes. Pour chacun, on calcule TPR, FPR et precision.

In [11]:
var scored = model.Transform(testData);
var scores = scored.GetColumn<float>("Score").ToArray();
var labels = scored.GetColumn<float>("Label").ToArray();

int totalAnom = labels.Count(l => l >= 0.5f);
int totalNorm = labels.Length - totalAnom;
float smin = scores.Min(), smax = scores.Max();

Console.WriteLine($"=== Trade-off seuil : TPR (detection) vs FPR (fausse alarme) ===");
Console.WriteLine($"  ({totalAnom} vraies anomalies, {totalNorm} vrais normaux ; scores de {smin:F3} a {smax:F3})");
Console.WriteLine($"  Seuil | TPR(detect) | FPR(fausse alarme) | Precision");
Console.WriteLine($"  ------+-------------+--------------------+----------");
for (int t = 1; t <= 9; t++)
{
    float thr = smin + (smax - smin) * t / 10f;
    int tp = 0, fp = 0;
    for (int i = 0; i < scores.Length; i++)
    {
        bool flagged = scores[i] > thr;
        if (flagged && labels[i] >= 0.5f) tp++;
        else if (flagged && labels[i] < 0.5f) fp++;
    }
    double tpr = totalAnom > 0 ? (double)tp / totalAnom : 0;
    double fpr = totalNorm > 0 ? (double)fp / totalNorm : 0;
    double prec = (tp + fp) > 0 ? (double)tp / (tp + fp) : 0;
    Console.WriteLine($"  {thr,5:F2} | {tpr,11:F2} | {fpr,18:F2} | {prec,10:F2}");
}


=== Trade-off seuil : TPR (detection) vs FPR (fausse alarme) ===


  (40 vraies anomalies, 120 vrais normaux ; scores de 0,001 a 0,513)


  Seuil | TPR(detect) | FPR(fausse alarme) | Precision


  ------+-------------+--------------------+----------


   0,05 |        1,00 |               0,41 |       0,45


...

   0,26 |        0,03 |               0,03 |       0,20


   0,31 |        0,03 |               0,01 |       0,50


   0,36 |        0,03 |               0,01 |       0,50


   0,41 |        0,03 |               0,01 |       0,50


   0,46 |        0,03 |               0,00 |       1,00


### Interpretation : un compromis detection / fausse alarme

Le sweep dessine la courbe ROC de facon lisible :

| Seuil | TPR (detection) | FPR (fausse alarme) | Precision | Lecture |
|-------|-----------------|---------------------|-----------|---------|
| 0,05 | 1,00 | 0,41 | 0,45 | on rattrape tout, mais 41 % de fausses alarmes |
| **0,10** | **0,93** | **0,23** | **0,58** | **bon compromis operationnel** |
| 0,21 | 0,12 | 0,05 | 0,45 | FPR < 5 %, mais on rate 88 % des anomalies |
| 0,46 | 0,03 | 0,00 | 1,00 | precis mais quasi inutile (3 % de detection) |

Le seuil **0,10** est un point de fonctionnement raisonnable : on detecte 93 % des anomalies pour 23 % de fausses alarmes. Si le metier exige `FPR <= 5 %`, il faut monter a ~0,21 — mais on accepte alors de rater la plupart des anomalies. Ce compromis n'a pas de reponse universelle : il depend du **cout relatif** d'une casse ratee (très eleve en maintenance predictive) face au cout d'une intervention inutile. L'[Exercice 2](#Exercice-2-:-Accorder-le-seuil-a-un-cout-operationnel) formalise cette accord sur une contrainte `TPR >= 0,95`.

## Exercice 1 : Effet du rang PCA sur l'AUC

Le rang de la PCA controle la **sensibilite** du detecteur. Avec 3 features :

- `rank: 1` -> sous-espace 1D, residu sur 2 dimensions (detecteur **tres sensible** : beaucoup de normaux flagges),
- `rank: 2` -> sous-espace plan, residu sur 1 dimension (compromis utilise dans le notebook),
- `rank: 3` -> tout est explique, residu nul (**aucune detection possible**).

**Objectifs** :
1. Boucler sur `rank = 1, 2, 3`, entrainer un `RandomizedPca` pour chaque valeur
2. Evaluer l'AUC sur `testData`
3. Verifier que `rank: 3` donne un AUC ~0.5 (aucun pouvoir de detection) et comparer 1 vs 2

**Indice** : recopiez le pipeline ci-dessus en parametrant `rank`. Pour `rank: 3`, les scores sont tous quasi nuls : la PCA a tout explique, il ne reste plus de residu a mesurer.

In [12]:
// Exercice 1 : Effet du rang PCA sur l'AUC
// TODO etudiant : boucle sur rank = 1, 2, 3, entraine un RandomizedPca pour chaque, evalue l'AUC.
// Indice : recopie le pipeline ci-dessus en parametrant rank. Pour rank:3, les scores sont ~0
//          (la PCA a tout explique) -> AUC ~0.5 (aucun pouvoir de detection).
// Etape 1 : pour rank dans {1, 2, 3}, construis le pipeline Concatenate -> NormalizeMinMax -> RandomizedPca(rank)
// Etape 2 : entraine sur trainData, transforme testData
// Etape 3 : evalue l'AUC (mlContext.AnomalyDetection.Evaluate)
// Etape 4 : affiche rank | AUC et conclus sur le compromis sensibilite / pouvoir discriminant
Console.WriteLine("Exercice 1 a completer");


Exercice 1 a completer


## Exercice 2 : Accorder le seuil a un cout operationnel

On veut un detecteur qui **rate au plus 5 % des anomalies** (TPR >= 0.95) tout en **minimisant les fausses alarmes**. Le sweep ci-dessus est trop grossier (10 seuils reguliers).

**Objectifs** :
1. Calculer une **courbe ROC fine** : 50 seuils repartis entre `smin` et `smax`, TPR et FPR pour chacun
2. Identifier le seuil qui realise **TPR >= 0.95 avec FPR minimal**
3. Reporter ce seuil operationnel et sa precision

**Indice** : meme boucle que le sweep, mais avec 50 seuils ; parcourir les couples (TPR, FPR) et garder le premier seuil (du plus eleve au plus bas) qui atteint TPR >= 0.95.

In [13]:
// Exercice 2 : Accorder le seuil a un cout operationnel
// TODO etudiant : courbe ROC fine (50 seuils), trouver le seuil a TPR >= 0.95 et FPR minimal.
//   TPR = TP / totalAnom   ;   FPR = FP / totalNorm
// Indice : 50 seuils entre smin et smax ; garde le seuil (le plus eleve d'abord) qui atteint TPR>=0.95.
// Etape 1 : boucle 50 seuils, calcule TPR + FPR pour chacun
// Etape 2 : cherche le seuil realisant TPR >= 0.95 avec FPR minimal
// Etape 3 : affiche le seuil operationnel, son FPR et sa precision
Console.WriteLine("Exercice 2 a completer");


Exercice 2 a completer


## Exercice 3 : Anomalies subtiles (limite de l'approche PCA)

Les anomalies actuelles sont decalees de ~3 ecarts-types : faciles a detecter. Que se passe-t-il avec des anomalies **subtiles** (~1.5 sigma), qui se rapprochent du regime normal ?

**Objectifs** :
1. Regenerer `testAnomalies` avec un decalage reduit (ex. Temperature ~ N(2, 2), Pressure ~ N(0.25, 0.1), Vibration ~ N(1.0, 0.8))
2. Reconstituer `testData`, re-evaluer l'AUC du modele (sans re-entrainer)
3. Constater la chute de l'AUC et conclure sur la **limite** de la detection PCA

**Indice** : plus les anomalies se rapprochent du regime normal, plus elles entrent dans le nuage appris et plus leur residu ressemble a celui d'un point normal. L'AUC chute vers 0.5. La PCA ne detecte bien que les anomalies **orthogonales** au sous-espace normal ; les anomalies « inline » (le long d'une direction normale) lui echappent.

In [14]:
// Exercice 3 : Anomalies subtiles (limite de l'approche PCA)
// TODO etudiant : regenere des anomalies subtiles (~1.5 sigma) et re-evalue l'AUC.
// Indice : anomalies plus proches du regime normal -> residu ressemblant a un point normal -> AUC chute.
//          La PCA ne detecte bien que les anomalies ORTHOGONALES au sous-espace normal (ici : un pic de pression).
// Etape 1 : regenere testAnomalies avec decalage reduit (Temperature~2, Pressure~0.25, Vibration~1.0)
// Etape 2 : reconstitue testData, re-evalue l'AUC du modele existant (sans re-fit)
// Etape 3 : compare l'AUC a la valeur du notebook et conclus sur la limite de la detection PCA
Console.WriteLine("Exercice 3 a completer");


Exercice 3 a completer


## Resume

Ce notebook a introduit la **detection d'anomalies non-supervisee** avec Randomized PCA :

| Concept | Description |
|---------|-------------|
| Detection d'anomalies | Apprentissage **non-supervise** : on apprend le regime normal, on teste si un point s'en ecarte |
| `AnomalyDetectionCatalog` | Catalogue ML.NET pour Randomized PCA (`mlContext.AnomalyDetection.Trainers.RandomizedPca`) |
| Score d'anomalie | **Residu de reconstruction** dans le sous-espace PCA (plus eleve = plus anomalous) |
| `rank` | Dimension du sous-espace ; controle la sensibilite (rank=n_features -> aucune detection) |
| AUC | **Separation** normaux/anormaux (1.0 = parfait, 0.5 = hasard) |
| DetectionRate@K FP | Fraction des anomalies detectee avant *K* fausses alarmes (metrique operationnelle) |
| Seuil de decision | Arbitre **TPR vs FPR** ; s'accorde au cout operationnel, pas au seuil par defaut |

**Points cles** :

- Comme en [ML-8](ML-8-Clustering.ipynb), le Fit est **non-supervise** : on s'entraine sur des donnees **normales** uniquement. Mais contrairement au clustering, l'Evaluate exploite des **etiquettes** (Label 0/1) pour calculer l'AUC.
- Le **rang** est un piege : `rank` ne peut pas depasser le nombre de features, et `rank = n_features` annule toute detection (residu nul). Le defaut (20) planterait ici.
- Le **seuil par defaut** suppose ~10 % de contamination ; sur un jeu d'entrainement integralement normal, il est trop bas et faut tuner (Exercice 2).
- La PCA detecte les anomalies **orthogonales** au sous-espace normal ; les anomalies « inline » subtiles lui echappent (Exercice 3) — c'est sa limite structurelle.


## References

**Algorithme (PCA pour la detection d'anomalies)**

- Lakhina, A., Crovella, M., & Diot, C. (2004). *Characterization of network-wide anomalies in traffic flows*. Proceedings of the 4th ACM SIGCOMM conference on Internet measurement, 201-206. --- la PCA appliquee a la detection d'anomalies (residu de reconstruction), origine du paradigme.
- Pfiefer, D., Sveridov, A., & Zabolotin, A. (2011). *Randomized PCA for fast anomaly detection*. --- l'algorithme randomise (SVD tronquee stochastique) implante par ML.NET.
- Jolliffe, I. T., & Cadima, J. (2016). *Principal component analysis: a review and recent developments*. Philosophical Transactions of the Royal Society A, 374(2065), 20150202. --- revue de reference sur la PCA.

**Metriques d'evaluation (ROC et detection rate)**

- Fawcett, T. (2006). *An introduction to ROC analysis*. Pattern Recognition Letters, 27(8), 861-874. --- l'AUC et la courbe ROC, lues dans le contexte de la detection.
- Aggarwal, C. C. (2017). *Outlier Analysis* (2nd ed.). Springer. --- cadre general de la detection d'anomalies (distance, densite, angle, reconstruction), ou la PCA est une des methodes de reconstruction.

**Cas d'usage (maintenance predictive)**

- Lei, Y., Li, N., Guo, L., Li, N., Yan, T., & Lin, J. (2018). *Machinery health prognostics: A systematic review from data acquisition to RUL prediction*. Mechanical Systems and Signal Processing, 104, 799-834. --- les capteurs industriels (temperature, pression, vibration) comme signaux de defaillance.
